# Tokyo Electricity Demand: SQL Analysis (SQLite)

A relational database built from real data, analyzed with SQL.

**Data:** 39,744 hourly records from 1 Apr 2022 to 12 Oct 2026 (JST).
* **TEPCO Power Grid "でんき予報":** demand, forecast, supply capacity and solar output.
* **Japan Meteorological Agency:** hourly temperature, humidity and sunshine at 8 Kanto stations.

**Steps**
1. Create the database (`schema.sql`) and insert the data (`build_database.py`).
2. Check the tables.
3. Run 16 queries using SELECT, WHERE, ORDER BY, GROUP BY / HAVING, JOIN, aggregate functions, CASE, subqueries, CTEs and window functions.

## 1. Create the database and insert the data

In [1]:
import sqlite3, pandas as pd
pd.set_option("display.width", 200)
import build_database, run_queries as R

print(open("schema.sql", encoding="utf-8").read())

-- =====================================================================================
-- Tokyo electricity demand database (SQLite)
-- 4 related tables: hourly demand, hourly weather (8 JMA stations), stations, calendar
-- =====================================================================================

DROP TABLE IF EXISTS weather_hourly;
DROP TABLE IF EXISTS demand_hourly;
DROP TABLE IF EXISTS stations;
DROP TABLE IF EXISTS calendar;

-- One row per day: Japanese calendar information
CREATE TABLE calendar (
    date          TEXT    PRIMARY KEY,          -- 'YYYY-MM-DD'
    year          INTEGER NOT NULL,
    month         INTEGER NOT NULL CHECK (month BETWEEN 1 AND 12),
    weekday       TEXT    NOT NULL,             -- 'Monday' ... 'Sunday'
    season        TEXT    NOT NULL CHECK (season IN ('Winter', 'Spring', 'Summer', 'Autumn')),
    holiday_name  TEXT,                         -- NULL on ordinary days
    is_holiday    INTEGER NOT NULL CHECK (is_holiday IN (0, 1)),
    

In [2]:
info = build_database.build()      # runs schema.sql, then INSERTs every row from the CSV files
info

{'rows': {'calendar': 1656,
  'stations': 8,
  'demand_hourly': 39744,
  'weather_hourly': 317952},
 'foreign_key_problems': 0}

## 2. Check the tables

In [3]:
con = sqlite3.connect("tokyo_electricity.db")
def sql(q):
    return pd.read_sql_query(q, con)

sql("""SELECT name AS table_name,
       (SELECT COUNT(*) FROM pragma_table_info(name)) AS columns
FROM sqlite_master WHERE type = 'table' ORDER BY name""")

,table_name,columns
0,calendar,0
1,demand_hourly,0
2,stations,0
3,weather_hourly,0


In [4]:
sql("SELECT * FROM stations")

,station_id,name_en,name_ja,prefecture,prefecture_population_10k
0,47615,Utsunomiya,宇都宮,Tochigi,193.3
1,47624,Maebashi,前橋,Gunma,193.9
2,47626,Kumagaya,熊谷,Saitama,734.5
3,47629,Mito,水戸,Ibaraki,286.7
4,47638,Kofu,甲府,Yamanashi,81.0
5,47662,Tokyo,東京,Tokyo,1404.8
6,47670,Yokohama,横浜,Kanagawa,923.7
7,47682,Chiba,千葉,Chiba,628.4


In [5]:
sql("SELECT * FROM calendar WHERE holiday_name IS NOT NULL LIMIT 5")

,date,year,month,weekday,season,holiday_name,is_holiday,is_workday
0,2022-04-29,2022,4,Friday,Spring,Showa Day,1,0
1,2022-05-03,2022,5,Tuesday,Spring,Constitution Day,1,0
2,2022-05-04,2022,5,Wednesday,Spring,Greenery Day,1,0
3,2022-05-05,2022,5,Thursday,Spring,Children's Day,1,0
4,2022-07-18,2022,7,Monday,Summer,Marine Day,1,0


In [6]:
sql("""SELECT d.datetime, d.demand_gw, d.solar_gw, s.name_en, w.temp_c
FROM demand_hourly d
JOIN weather_hourly w ON w.datetime = d.datetime
JOIN stations s ON s.station_id = w.station_id
WHERE d.datetime = '2022-08-02 13:00:00'
ORDER BY w.temp_c DESC""")

,datetime,demand_gw,solar_gw,name_en,temp_c
0,2022-08-02 13:00:00,59.3,11.34,Kumagaya,38.25
1,2022-08-02 13:00:00,59.3,11.34,Maebashi,37.60
2,2022-08-02 13:00:00,59.3,11.34,Kofu,37.30
3,2022-08-02 13:00:00,59.3,11.34,Utsunomiya,36.40
4,2022-08-02 13:00:00,59.3,11.34,Mito,36.15
5,2022-08-02 13:00:00,59.3,11.34,Tokyo,34.90
6,2022-08-02 13:00:00,59.3,11.34,Yokohama,34.70
7,2022-08-02 13:00:00,59.3,11.34,Chiba,33.50


## 3. Analysis queries
Each query is stored in `queries.sql` and repeated here with its result.

### A. SELECT, WHERE, ORDER BY

**q01_top10_demand_hours**: What were the 10 highest-demand hours, and how hot was Tokyo then?

In [7]:
sql("""
SELECT d.datetime,
       d.demand_gw,
       w.temp_c AS tokyo_temp_c,
       d.usage_rate_pct
FROM demand_hourly AS d
JOIN weather_hourly AS w
  ON w.datetime = d.datetime AND w.station_id = 47662          -- Tokyo station
ORDER BY d.demand_gw DESC
LIMIT 10;
""")

,datetime,demand_gw,tokyo_temp_c,usage_rate_pct
0,2022-08-02 13:00:00,59.30,34.90,92
1,2022-08-02 14:00:00,59.21,35.20,93
2,2022-08-03 13:00:00,58.89,34.45,93
3,2022-08-02 15:00:00,58.50,34.80,95
4,2022-08-03 14:00:00,58.48,32.85,95
5,2022-08-03 11:00:00,58.13,35.10,95
6,2022-08-03 12:00:00,58.01,35.05,95
7,2022-08-02 12:00:00,57.96,34.60,93
8,2022-08-02 11:00:00,57.83,34.35,94
9,2022-08-02 16:00:00,57.55,34.35,97


**q02_tight_supply_hours**: Which hours came closest to running out of supply (usage rate of 97% or more)?

In [8]:
sql("""
SELECT datetime, demand_gw, supply_gw, usage_rate_pct,
       ROUND(supply_gw - demand_gw, 2) AS spare_gw
FROM demand_hourly
WHERE usage_rate_pct >= 97
ORDER BY usage_rate_pct DESC, spare_gw ASC;
""")

,datetime,demand_gw,supply_gw,usage_rate_pct,spare_gw
0,2026-07-22 17:00:00,53.52,53.89,99,0.37
1,2025-12-14 09:00:00,38.46,38.90,98,0.44
2,2025-12-14 10:00:00,38.91,39.36,98,0.45
3,2026-01-03 09:00:00,34.66,35.32,98,0.66
4,2024-09-10 16:00:00,49.79,50.49,98,0.70
5,2024-09-11 16:00:00,51.75,52.62,98,0.87
6,2024-09-12 16:00:00,51.60,52.63,98,1.03
7,2026-05-04 16:00:00,24.23,24.79,97,0.56
8,2026-05-01 10:00:00,30.70,31.37,97,0.67
9,2026-01-03 08:00:00,34.84,35.70,97,0.86


### B. Aggregate functions

**q03_overall_summary**: What does the whole dataset look like in one row?

In [9]:
sql("""
SELECT COUNT(*)                                  AS hours,
       MIN(date)                                 AS first_day,
       MAX(date)                                 AS last_day,
       ROUND(AVG(demand_gw), 2)                  AS avg_demand_gw,
       MIN(demand_gw)                            AS min_demand_gw,
       MAX(demand_gw)                            AS max_demand_gw,
       ROUND(SUM(demand_gw) / 1000.0, 1)         AS total_energy_twh,
       ROUND(SUM(solar_gw) / SUM(demand_gw) * 100, 1) AS solar_share_pct
FROM demand_hourly;
""")

,hours,first_day,last_day,avg_demand_gw,min_demand_gw,max_demand_gw,total_energy_twh,solar_share_pct
0,39744,2022-04-01,2026-10-12,31.71,18.53,59.3,1260.3,9.0


### C. GROUP BY and HAVING

**q04_by_year**: How did demand, energy use and solar change from year to year?

In [10]:
sql("""
SELECT c.year,
       COUNT(*)                                    AS hours,
       ROUND(AVG(d.demand_gw), 2)                  AS avg_demand_gw,
       MAX(d.demand_gw)                            AS peak_demand_gw,
       ROUND(SUM(d.demand_gw) / 1000.0, 1)         AS energy_twh,
       MAX(d.solar_gw)                             AS max_solar_gw,
       ROUND(SUM(d.solar_gw) / SUM(d.demand_gw) * 100, 1) AS solar_share_pct
FROM demand_hourly AS d
JOIN calendar AS c ON c.date = d.date
GROUP BY c.year
ORDER BY c.year;
""")

,year,hours,avg_demand_gw,peak_demand_gw,energy_twh,max_solar_gw,solar_share_pct
0,2022,6600,31.52,59.30,208.0,15.156,8.2
1,2023,8760,31.65,55.25,277.3,16.006,9.2
2,2024,8784,32.06,56.99,281.6,16.634,8.8
3,2025,8760,32.17,57.54,281.8,16.518,9.2
4,2026,6840,30.94,56.42,211.6,17.764,9.6


**q05_high_demand_months**: Which calendar months average more than 33 GW? (GROUP BY + HAVING)

In [11]:
sql("""
SELECT c.month,
       ROUND(AVG(d.demand_gw), 2) AS avg_demand_gw,
       MAX(d.demand_gw)           AS peak_demand_gw,
       COUNT(DISTINCT d.date)     AS days
FROM demand_hourly AS d
JOIN calendar AS c ON c.date = d.date
GROUP BY c.month
HAVING AVG(d.demand_gw) > 33
ORDER BY avg_demand_gw DESC;
""")

,month,avg_demand_gw,peak_demand_gw,days
0,7,36.62,56.99,155
1,8,36.54,59.30,155
2,1,35.33,51.37,124
3,2,35.03,51.79,113
4,12,33.38,46.85,124
5,9,33.02,55.21,150


**q06_peak_hour_by_season**: At what time of day does demand peak in each season?

In [12]:
sql("""
SELECT c.season,
       d.hour,
       ROUND(AVG(d.demand_gw), 2) AS avg_demand_gw
FROM demand_hourly AS d
JOIN calendar AS c ON c.date = d.date
GROUP BY c.season, d.hour
HAVING AVG(d.demand_gw) = (
    SELECT MAX(avg_h) FROM (
        SELECT AVG(d2.demand_gw) AS avg_h
        FROM demand_hourly AS d2
        JOIN calendar AS c2 ON c2.date = d2.date
        WHERE c2.season = c.season
        GROUP BY d2.hour))
ORDER BY avg_demand_gw DESC;
""")

,season,hour,avg_demand_gw
0,Summer,13,42.06
1,Winter,18,39.09
2,Autumn,14,34.66
3,Spring,18,31.60


### D. JOINs with the calendar

**q07_workday_vs_dayoff**: How much more electricity is used on workdays than on weekends and holidays?

In [13]:
sql("""
SELECT CASE WHEN c.is_workday = 1 THEN 'Workday' ELSE 'Weekend / holiday' END AS day_type,
       COUNT(DISTINCT c.date)       AS days,
       ROUND(AVG(d.demand_gw), 2)   AS avg_demand_gw,
       MAX(d.demand_gw)             AS peak_demand_gw
FROM demand_hourly AS d
JOIN calendar AS c ON c.date = d.date
GROUP BY day_type
ORDER BY avg_demand_gw DESC;
""")

,day_type,days,avg_demand_gw,peak_demand_gw
0,Workday,1114,33.06,59.30
1,Weekend / holiday,542,28.94,47.72


**q08_by_weekday**: Which day of the week uses the most and least electricity (public holidays excluded)?

In [14]:
sql("""
SELECT c.weekday,
       ROUND(AVG(d.demand_gw), 2) AS avg_demand_gw,
       MAX(d.demand_gw)           AS peak_demand_gw
FROM demand_hourly AS d
JOIN calendar AS c ON c.date = d.date
WHERE c.is_holiday = 0
GROUP BY c.weekday
ORDER BY avg_demand_gw DESC;
""")

,weekday,avg_demand_gw,peak_demand_gw
0,Tuesday,33.40,59.30
1,Wednesday,33.38,58.89
2,Thursday,33.06,56.28
3,Friday,32.80,55.46
4,Monday,32.60,57.44
5,Saturday,29.61,47.72
6,Sunday,28.48,47.50


**q09_holidays_ranked**: Which holidays have the lowest demand?

In [15]:
sql("""
SELECT c.holiday_name,
       COUNT(DISTINCT c.date)     AS days,
       ROUND(AVG(d.demand_gw), 2) AS avg_demand_gw,
       MIN(d.demand_gw)           AS lowest_hour_gw
FROM calendar AS c
JOIN demand_hourly AS d ON d.date = c.date
WHERE c.holiday_name IS NOT NULL
GROUP BY c.holiday_name
ORDER BY avg_demand_gw ASC;
""")

,holiday_name,days,avg_demand_gw,lowest_hour_gw
0,Greenery Day,5,22.36,18.53
1,Constitution Day,5,22.38,18.65
2,Children's Day,5,22.41,18.55
3,Showa Day,5,24.34,19.80
4,Sports Day,5,25.74,20.14
5,Culture Day,4,25.75,20.80
6,New Year's Day,4,26.32,22.29
7,Autumnal Equinox Day,5,27.59,22.07
8,Substitute Holiday,10,28.29,19.07
9,Labor Thanksgiving Day,4,28.34,23.21


### E. JOINs with weather and stations

**q10_station_records**: Which of the 8 Kanto weather stations is the hottest and the coldest?

In [16]:
sql("""
SELECT s.name_en, s.name_ja, s.prefecture,
       ROUND(AVG(w.temp_c), 2)                          AS avg_temp_c,
       MAX(w.temp_c)                                    AS max_temp_c,
       MIN(w.temp_c)                                    AS min_temp_c,
       SUM(CASE WHEN w.temp_c >= 35 THEN 1 ELSE 0 END)  AS hours_35c_plus,
       SUM(CASE WHEN w.temp_c < 0 THEN 1 ELSE 0 END)    AS hours_below_0c
FROM weather_hourly AS w
JOIN stations AS s ON s.station_id = w.station_id
GROUP BY s.station_id
ORDER BY max_temp_c DESC;
""")

,name_en,name_ja,prefecture,avg_temp_c,max_temp_c,min_temp_c,hours_35c_plus,hours_below_0c
0,Maebashi,前橋,Gunma,17.28,40.05,-5.30,499,560
1,Kumagaya,熊谷,Saitama,17.61,40.05,-5.65,644,534
2,Kofu,甲府,Yamanashi,17.11,39.15,-7.95,392,1381
3,Tokyo,東京,Tokyo,17.96,38.05,-3.15,136,80
4,Utsunomiya,宇都宮,Tochigi,16.47,37.25,-6.90,142,1522
5,Mito,水戸,Ibaraki,16.46,37.00,-6.00,69,1170
6,Yokohama,横浜,Kanagawa,18.27,36.95,-3.10,54,44
7,Chiba,千葉,Chiba,18.33,36.40,-1.50,36,49


**q11_demand_by_temperature_band**: How does demand change with Tokyo's temperature? (CASE bands + JOIN + GROUP BY)

In [17]:
sql("""
SELECT CASE
         WHEN w.temp_c < 5  THEN '1: below 5 °C'
         WHEN w.temp_c < 10 THEN '2: 5–10 °C'
         WHEN w.temp_c < 15 THEN '3: 10–15 °C'
         WHEN w.temp_c < 20 THEN '4: 15–20 °C'
         WHEN w.temp_c < 25 THEN '5: 20–25 °C'
         WHEN w.temp_c < 30 THEN '6: 25–30 °C'
         WHEN w.temp_c < 35 THEN '7: 30–35 °C'
         ELSE                    '8: 35 °C and above'
       END                         AS temp_band,
       COUNT(*)                    AS hours,
       ROUND(AVG(d.demand_gw), 2)  AS avg_demand_gw,
       MAX(d.demand_gw)            AS peak_demand_gw,
       MIN(d.demand_gw)            AS low_demand_gw
FROM demand_hourly AS d
JOIN weather_hourly AS w ON w.datetime = d.datetime
JOIN stations AS s ON s.station_id = w.station_id
WHERE s.name_en = 'Tokyo'
GROUP BY temp_band
ORDER BY temp_band;
""")

,temp_band,hours,avg_demand_gw,peak_demand_gw,low_demand_gw
0,1: below 5 °C,2703,34.37,51.79,23.56
1,2: 5–10 °C,5859,33.91,47.61,21.19
2,3: 10–15 °C,6472,29.40,42.96,18.61
3,4: 15–20 °C,7151,26.76,38.16,18.53
4,5: 20–25 °C,7685,28.46,43.75,19.27
5,6: 25–30 °C,7172,34.08,52.68,22.79
6,7: 30–35 °C,2566,45.56,59.30,28.30
7,8: 35 °C and above,136,51.90,59.21,41.08


**q12_kanto_weighted_temp_on_peak_days**: On the 5 highest-demand days, how hot was it across Kanto (population-weighted)?

In [18]:
sql("""
WITH daily_peak AS (
    SELECT date, MAX(demand_gw) AS peak_gw
    FROM demand_hourly
    GROUP BY date
    ORDER BY peak_gw DESC
    LIMIT 5
)
SELECT p.date,
       c.weekday,
       p.peak_gw,
       ROUND(SUM(w.temp_c * s.prefecture_population_10k) / SUM(s.prefecture_population_10k), 1)
                                AS kanto_weighted_max_temp_c,
       MAX(w.temp_c)            AS hottest_station_temp_c
FROM daily_peak AS p
JOIN calendar AS c ON c.date = p.date
JOIN demand_hourly AS d ON d.date = p.date AND d.demand_gw = p.peak_gw   -- the peak hour itself
JOIN weather_hourly AS w ON w.datetime = d.datetime
JOIN stations AS s ON s.station_id = w.station_id
GROUP BY p.date
ORDER BY p.peak_gw DESC;
""")

,date,weekday,peak_gw,kanto_weighted_max_temp_c,hottest_station_temp_c
0,2022-08-02,Tuesday,59.30,35.5,38.25
1,2022-08-03,Wednesday,58.89,35.1,37.95
2,2025-08-06,Wednesday,57.54,35.9,37.95
3,2022-08-01,Monday,57.44,34.7,37.65
4,2025-08-05,Tuesday,57.03,36.5,40.05


### F. Subqueries, CTEs and window functions

**q13_peak_day_each_year**: What was the single highest-demand day in each year, and what kind of day was it?

In [19]:
sql("""
WITH daily AS (
    SELECT d.date, c.year, c.weekday, c.holiday_name, MAX(d.demand_gw) AS peak_gw
    FROM demand_hourly AS d
    JOIN calendar AS c ON c.date = d.date
    GROUP BY d.date
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY year ORDER BY peak_gw DESC) AS rn
    FROM daily
)
SELECT year, date, weekday, peak_gw
FROM ranked
WHERE rn = 1
ORDER BY year;
""")

,year,date,weekday,peak_gw
0,2022,2022-08-02,Tuesday,59.30
1,2023,2023-07-18,Tuesday,55.25
2,2024,2024-07-29,Monday,56.99
3,2025,2025-08-06,Wednesday,57.54
4,2026,2026-07-22,Wednesday,56.42


**q14_days_above_55gw**: How many days reached 55 GW or more each year, and how hot did Tokyo get on them?

In [20]:
sql("""
SELECT c.year,
       COUNT(*)                         AS days_55gw_plus,
       ROUND(AVG(t.max_temp_c), 1)      AS avg_tokyo_max_temp_c
FROM (
    SELECT d.date, MAX(d.demand_gw) AS peak_gw, MAX(w.temp_c) AS max_temp_c
    FROM demand_hourly AS d
    JOIN weather_hourly AS w ON w.datetime = d.datetime AND w.station_id = 47662
    GROUP BY d.date
) AS t
JOIN calendar AS c ON c.date = t.date
WHERE t.peak_gw >= 55
GROUP BY c.year
ORDER BY c.year;
""")

,year,days_55gw_plus,avg_tokyo_max_temp_c
0,2022,5,35.1
1,2023,1,37.0
2,2024,2,36.5
3,2025,4,35.9
4,2026,2,35.5


**q15_forecast_accuracy_by_season**: How accurate is TEPCO's demand forecast in each season?

In [21]:
sql("""
SELECT c.season,
       COUNT(*)                                                    AS hours,
       ROUND(AVG(ABS(d.demand_gw - d.forecast_gw) / d.demand_gw) * 100, 2) AS mape_pct,
       ROUND(AVG(d.demand_gw - d.forecast_gw), 3)                  AS avg_bias_gw,
       ROUND(MAX(ABS(d.demand_gw - d.forecast_gw)), 2)             AS worst_miss_gw
FROM demand_hourly AS d
JOIN calendar AS c ON c.date = d.date
GROUP BY c.season
ORDER BY mape_pct DESC;
""")

,season,hours,mape_pct,avg_bias_gw,worst_miss_gw
0,Spring,10296,1.11,0.004,3.02
1,Summer,11040,1.05,-0.011,6.22
2,Winter,8664,0.95,-0.024,3.75
3,Autumn,9744,0.90,-0.019,3.82


**q16_solar_by_hour_spring**: How much of the demand does solar cover at each daytime hour in spring?

In [22]:
sql("""
SELECT d.hour,
       ROUND(AVG(d.demand_gw), 2)                          AS avg_demand_gw,
       ROUND(AVG(d.solar_gw), 2)                           AS avg_solar_gw,
       ROUND(AVG(d.demand_gw - d.solar_gw), 2)             AS avg_net_demand_gw,
       ROUND(SUM(d.solar_gw) / SUM(d.demand_gw) * 100, 1)  AS solar_share_pct
FROM demand_hourly AS d
JOIN calendar AS c ON c.date = d.date
WHERE c.season = 'Spring' AND d.hour BETWEEN 6 AND 18
GROUP BY d.hour
ORDER BY d.hour;
""")

,hour,avg_demand_gw,avg_solar_gw,avg_net_demand_gw,solar_share_pct
0,6,24.75,1.27,23.49,5.1
1,7,26.37,3.55,22.82,13.5
2,8,28.82,6.18,22.65,21.4
3,9,30.71,8.41,22.30,27.4
4,10,31.24,9.94,21.30,31.8
5,11,31.40,10.59,20.82,33.7
6,12,30.48,10.37,20.11,34.0
7,13,30.84,9.42,21.42,30.6
8,14,30.55,7.68,22.87,25.1
9,15,30.35,5.33,25.03,17.5


## Key findings
1. **Summer heat sets the record.** All 10 highest-demand hours fell on 2–3 Aug 2022, between 11:00 and 16:00, when Tokyo was at about 33–35 °C. The peak was 59.3 GW.
2. **Temperature works both ways.** Average demand is lowest at 15–20 °C (26.8 GW) and almost doubles at 35 °C and above (51.9 GW). Cold weather below 5 °C also raises it, to 34.4 GW.
3. **The calendar matters.** Workdays average 33.1 GW, compared with 28.9 GW on weekends and holidays. Tuesday is the busiest day and Sunday the quietest. The three Golden Week holidays have the lowest demand of any holiday, at about 22.4 GW.
4. **Supply gets tight in every season, not only summer.** The tightest hour was 99 % usage, with only 0.37 GW spare, at 17:00 on 22 Jul 2026. Other hours at 97 % or more include winter mornings (14 Dec 2025, 3 Jan 2026) and early May.
5. **Inland stations are the hottest.** Kumagaya and Maebashi both reached 40.05 °C. Kumagaya had 644 hours at 35 °C or above, against 136 for Tokyo and 36 for Chiba.
6. **Solar is reshaping the day.** On spring days, solar covers 34 % of demand at noon, which lowers the demand left for other power plants to 20.1 GW. Spring also has TEPCO's least accurate forecasts (1.11 % error).